In [2]:
import sys
from pathlib import Path

RACINE = Path.cwd().parent
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

from src.outils import consulter_pdf

In [6]:
MODELE = "qwen3:4b"

llm = ChatOllama(model=MODELE, temperature=0)

INSTRUCTIONS = """Tu es un assistant documentaire.
Tu disposes de l'outil consulter_pdf, qui cherche dans l'article
"Attention Is All You Need".

Règles :
1. Réponds en français.
2. Pour toute question sur le contenu du document, appelle consulter_pdf
   avant de répondre.
3. Appuie-toi uniquement sur les résultats de l'outil et cite la page :
   [p. 3].
4. Si l'outil ne donne pas la réponse, dis-le clairement.
5. Maximum 3 appels à l'outil par question."""

agent = create_react_agent(
    model=llm,
    tools=[consulter_pdf],
    prompt=INSTRUCTIONS,
)
print("Agent créé avec le modèle :", MODELE)

Agent créé avec le modèle : qwen3:4b


C:\Users\bouho\AppData\Local\Temp\ipykernel_3988\2160803708.py:18: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [8]:
resultat = agent.invoke({
    "messages": [("user", "Explique le mécanisme d'attention multi-têtes.")]
})
print(resultat["messages"][-1].content)

Le mécanisme d'attention multi-têtes permet au modèle Transformer de diviser l'attention en plusieurs "têtes" (heads) indépendantes, chacune capable de capturer des aspects spécifiques de l'entrée. Selon [p. 2], cette approche permet de contrebalancer l'effet d'averaging (réduction de la résolution effective) lié à l'attention standard. Les exemples présentés sur [p. 14] et [p. 15] montrent que chaque tête d'attention peut être spécialisée pour des tâches distinctes, comme la résolution d'anaphores (ex. le mot "its") ou l'analyse de la structure de la phrase. Ainsi, l'attention multi-têtes permet au modèle de traiter diverses informations simultanément tout en améliorant la qualité des représentations.


In [ ]:
for message in resultat["messages"]:
    print(f"--- {message.type.upper()} ---")
    if getattr(message, "tool_calls", None):
        print("Appel d'outil :", message.tool_calls)
    else:
        print(str(message.content)[:300])

--- HUMAN ---
Explique le mécanisme d'attention multi-têtes.
--- AI ---
Appel d'outil : [{'name': 'consulter_pdf', 'args': {'k': 3, 'question': "Explique le mécanisme d'attention multi-têtes.", 'type_resultat': 'texte'}, 'id': '1c6a3855-4c00-4955-a7d4-1b0205d0b385', 'type': 'tool_call'}]
--- TOOL ---
{"question": "Explique le mécanisme d'attention multi-têtes.", "sources": [{"reference": "S1", "id": "bdfaa68d8984f0dc02beaca5:texte:2:8", "type": "texte", "page": 2, "contenu": "reduced effective resolution due to averaging attention - weighted positions , an effect we counteract with multi - head 
--- AI ---
Le mécanisme d'attention multi-têtes permet au modèle Transformer de diviser l'attention en plusieurs "têtes" (heads) indépendantes, chacune capable de capturer des aspects spécifiques de l'entrée. Selon [p. 2], cette approche permet de contrebalancer l'effet d'averaging (réduction de la résolution 
